In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
import numpy as np
import glob
import os

# List of models and corresponding directories
models = ['XGB', 'SVM', 'RF', 'DNN', 'ANN']

# Lists to store individual run results and summary statistics
all_results = []
summary_results = []

# Loop over each model
for model_name in models:
    # Construct the file search pattern for each model's results
    pattern = os.path.join(f"Results_{model_name}", f"{model_name}_results_*.csv")
    files = sorted(glob.glob(pattern))
    auc_list = []

    # Loop over each run file for the current model
    for f in files:
        df = pd.read_csv(f)
        # Convert Real_Class to binary format (1=Active, 0=Inactive)
        df['Real_Class_Bin'] = df['Real_Class'].apply(lambda x: 1 if x == 'Active' else 0)
        # Calculate ROC-AUC using the probability of being Active
        auc = roc_auc_score(df['Real_Class_Bin'], df['Active_Prob'])
        auc_list.append(auc)
        # Store individual run results
        all_results.append({'Model': model_name, 'File': f, 'AUC-ROC': auc})

    # Calculate mean and SEM (standard error of the mean) for ROC-AUC
    if auc_list:  # Check that the list is not empty
        avg_auc = np.mean(auc_list)
        sem_auc = np.std(auc_list, ddof=1) / np.sqrt(len(auc_list))
    else:
        avg_auc = np.nan
        sem_auc = np.nan

    # Store summary results for the current model
    summary_results.append({'Model': model_name, 'Average_AUC': avg_auc, 'SEM': sem_auc})

# Save individual run results to CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv("All_Model_AUC.csv", index=False)

# Save summary statistics (mean ± SEM) to CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv("All_Model_AUC_Summary.csv", index=False)

print("Individual run results saved in All_Model_AUC.csv")
print("Summary with mean ± SEM saved in All_Model_AUC_Summary.csv")
